# Module 2 POC Training -- FIRE + LongDRScreening + Tianjin

Runs `train_module2_poc.py` against all three Module 2 data sources at once: FIRE and
LongDRScreening (placeholder Stage-2 conditioning unless a Module 1 cache says otherwise,
same as notebook 04) plus Tianjin (real clinical DR grade conditioning, resolved per-eye --
see notebook 05 and `docs/IMPLEMENTATION_PLAN.md` Task B).

As of this revision, the Generator (`DRForestGAN-v2/base_model.py`) uses **AdaIN stage
conditioning** in its bottleneck (Task C, additive to the original channel-concat
conditioning), training pairs can use a **pre-registered, baseline-aligned follow-up image**
from `module1/train_registration.py` (Task E) instead of the raw one, and after training this
notebook runs **`evaluate_trajectory.py`** (Task D) to report per-cascade-step FID/PSNR/SSIM
against Tianjin's real follow-up images -- not just a single-step number. **Any checkpoint
from before this revision will NOT load into the new Generator** -- retrain from scratch.

**Run this after notebook 04** (FIRE/LongDR Module 1 caches) **and notebook 05** (Tianjin
Module 1 cache + laterality resolution) **have both finished** -- this notebook only reads
their cache outputs from Drive, it doesn't rebuild them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'main'  # <-- change if the code you need (e.g. this Tianjin work) isn't merged yet
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache')
for _name in ['module1_outputs_fire.pt', 'module1_outputs_longdr.pt', 'module1_outputs_tianjin.pt']:
    _p = os.path.join(CACHE_DIR, _name)
    assert os.path.isfile(_p), (
        f"Not found: {_p} -- run notebook 04 (FIRE/LongDR) and notebook 05 (Tianjin) first."
    )
    print('✓', _p)

In [ ]:
# Unzip the three raw datasets locally (train_module2_poc.py's loaders read images directly
# at training time, not just at cache-build time). Same unzip conventions as notebooks 01/04/05.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip" -d LongDRScreening_20150209
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found -- see notebook 05 for troubleshooting.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

print('FIRE Images present   :', os.path.isdir('/content/data/FIRE_dataset/FIRE/Images'))
print('LongDR norm present   :', os.path.isdir('/content/data/LongDRScreening_20150209/FundusImagesNormalized'))
print('Tianjin manifest found:', os.path.isfile(os.path.join(TIANJIN_DIR, 'corrected_manifest.csv')))

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch torchmetrics torch-fidelity opencv-python-headless

In [ ]:
# Laterality resolution (Task B.3): Organized_Data of Patients.xlsx grades DR per EYE
# (OS/OD), not per patient, and corrected_manifest.csv doesn't say which eye a row is --
# tianjin_dataset.py REQUIRES laterality_resolved.csv to exist in TIANJIN_DIR before it can
# run. Reuse a Drive-persisted copy if one already exists (resolving ~1,100 images takes a few
# minutes -- no need to redo it every session); otherwise compute it fresh here and persist it
# to Drive for next time.
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'laterality_resolved.csv')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    import shutil
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive: {LATERALITY_DRIVE_PATH}')
else:
    %cd {REPO_CODE_DIR}
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"
    # Read the printed same-eye disagreement rate above BEFORE trusting this for training --
    # see resolve_eye_laterality.py's module docstring and docs/IMPLEMENTATION_PLAN.md's
    # "Open decisions" section (>5% disagreement means the heuristic needs tuning, not
    # silent acceptance).
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    import shutil
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive: {LATERALITY_DRIVE_PATH}')

In [ ]:
# Task E: train the training-only affine registration network once, and pre-warp every
# baseline<->follow-up pair (all 3 sources) so train_module2_poc.py can condition on
# baseline-aligned follow-up images instead of raw ones. Regenerated each session (a few
# minutes at POC scale) -- consider persisting registration_net.ckpt/registered_followups.pt
# to Drive under module1_cache/ if this becomes a bottleneck.
%cd {REPO_CODE_DIR}/module1
!python train_registration.py \
    --fire-dir /content/data/FIRE_dataset \
    --longdr-dir /content/data/LongDRScreening_20150209 \
    --tianjin-dir "{TIANJIN_DIR}" \
    --num-epochs 10 \
    --checkpoint-out ./registration_net.ckpt \
    --cache-out ./registered_followups.pt

REGISTRATION_CACHE = f'{REPO_CODE_DIR}/module1/registered_followups.pt'

In [ ]:
# --fire-dir / --longdr-dir / --tianjin-dir MUST be passed explicitly: train_module2_poc.py's
# own argparse defaults resolve relative to the cloned repo's working directory, not where the
# data actually lives on the Colab VM (see combined_dataset.py / train_module2_poc.py docstrings).
%cd {REPO_CODE_DIR}
!python train_module2_poc.py \
    --fire-dir /content/data/FIRE_dataset \
    --longdr-dir /content/data/LongDRScreening_20150209 \
    --tianjin-dir "{TIANJIN_DIR}" \
    --fire-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_fire.pt \
    --longdr-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_longdr.pt \
    --tianjin-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_tianjin.pt \
    --registration-cache "{REGISTRATION_CACHE}" \
    --num-epochs 5

In [ ]:
from IPython.display import Image as IPImage, display
import glob, json, os

samples = sorted(glob.glob(f'{REPO_CODE_DIR}/training_samples_poc/epoch_*.jpg'))
if samples:
    display(IPImage(samples[-1]))
else:
    print('No training_samples_poc/epoch_*.jpg found -- the training cell above did not finish. Read its output.')

rp = f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/models_poc/poc_results.json'
print(json.dumps(json.load(open(rp)), indent=2) if os.path.isfile(rp) else f'{rp} not found')

In [ ]:
# Task D's required evaluation addition: per-cascade-step FID/PSNR/SSIM against Tianjin's real
# follow-up images (the only source with a real grade on BOTH baseline and follow-up -- FIRE/
# LongDR can't participate in this, see evaluate_trajectory.py's module docstring). This is
# what actually answers RQ1's "how does credibility vary across stage transitions" question --
# the single-step metrics from the training cell above do not.
%cd {REPO_CODE_DIR}
!python evaluate_trajectory.py \
    --tianjin-dir "{TIANJIN_DIR}" \
    --tianjin-module1-cache /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_tianjin.pt \
    --generator-checkpoint ./DRForestGAN-v2/stargan/models_poc/final-G.ckpt \
    --out ./trajectory_eval_results.json

import json as _json
print(_json.dumps(_json.load(open(f'{REPO_CODE_DIR}/trajectory_eval_results.json')), indent=2))

## Honest caveats for the presentation

- Same POC-scale caveats as notebook 04 (5 epochs, manuscript-matched Adam settings, EX+MA-only
  segmentation conditioning).
- Tianjin adds the pipeline's first *real* clinical grade conditioning source, now correctly
  resolved per-eye (Task B) -- but the ETDRS(1-5)->ICDR(0-4) offset, while confirmed correct
  against the column legend, is not the same claim as ETDRS/ICDR being clinically equivalent
  staging *criteria*; that still wants Dr. Atienza/adviser sign-off (see `tianjin_dataset.py`).
- FIRE and LongDR still fall back to placeholder Stage-2 conditioning wherever no Module 1
  cache entry exists for a given image -- this run does not change that.
- AdaIN stage conditioning (Task C), autoregressive multi-stage cascade generation (Task D),
  and a training-only registration network (Task E) are now implemented -- the manuscript's
  claimed original contributions on top of the verified DRForecastGAN base. The per-cascade-
  step numbers above come from very few real Tianjin progression pairs at POC scale; read the
  n_pairs in each bucket before treating any step's FID/PSNR/SSIM as reliable, and note that
  buckets with too few pairs report FID as null rather than a misleading number (see
  `evaluate_trajectory.py`).
- The registration network's warp quality has not been visually audited here -- inspect a few
  `registered_followups.pt` entries before trusting that pre-registration measurably helped,
  rather than assuming it did because the mechanism now exists.